In [12]:
from pathlib import Path
import pandas as pd
from credit_card_fraud_detection.paths import ROOT, RAW
from datetime import date

In [13]:
train = RAW / "train.csv"
test = RAW / "test.csv"

In [14]:
train_df = pd.read_csv(train)
test_df = pd.read_csv(test)

## Structure and Tidiness

In [16]:
# standardize column names
train_df.columns = (train_df.columns.str.strip().str.lower()
            .str.replace(r'[^0-9a-z]+', "_", regex=True).str.strip("_"))
test_df.columns = (test_df.columns.str.strip().str.lower()
            .str.replace(r'[^0-9a-z]+', "_", regex=True).str.strip("_"))

print(train_df.columns)
print(test_df.columns)

Index(['id', 'amount_usd', 'hours_since_last_txn', 'txn_count_last_24h',
       'distance_from_home_km', 'card_age_months', 'customer_age',
       'account_balance_usd', 'cvv_retry_count', 'velocity_score',
       'time_of_day_hour', 'day_of_week', 'merchant_risk_score',
       'prior_disputes', 'merchant_category', 'card_type', 'auth_method',
       'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant',
       'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch',
       'is_ai_generated_scam_attempt', 'is_fraud'],
      dtype='object')
Index(['id', 'amount_usd', 'hours_since_last_txn', 'txn_count_last_24h',
       'distance_from_home_km', 'card_age_months', 'customer_age',
       'account_balance_usd', 'cvv_retry_count', 'velocity_score',
       'time_of_day_hour', 'day_of_week', 'merchant_risk_score',
       'prior_disputes', 'merchant_category', 'card_type', 'auth_method',
       'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant',
       

In [19]:
for c in train_df.select_dtypes("object"):
    print(train_df[c].value_counts(dropna=False).head(10))
    print(train_df[c].value_counts().tail(10))

merchant_category
Online Retail    155097
Groceries        142578
Restaurants      101360
Fuel              59880
Electronics       48693
Streaming         47196
Utilities         42027
Healthcare        35889
Travel            28715
Gaming            23784
Name: count, dtype: int64
merchant_category
Restaurants        101360
Fuel                59880
Electronics         48693
Streaming           47196
Utilities           42027
Healthcare          35889
Travel              28715
Gaming              23784
Gift Cards           9456
Crypto Exchange      5285
Name: count, dtype: int64
card_type
Visa          345001
Mastercard    254857
Amex           45797
RuPay          37070
Discover       17235
Name: count, dtype: int64
card_type
Visa          345001
Mastercard    254857
Amex           45797
RuPay          37070
Discover       17235
Name: count, dtype: int64
auth_method
3D Secure            210036
OTP                  175178
PIN                  154020
Biometric             93666
No Aut

## Outliers (detect, classify, treat)

In [ ]:
numerical_cols = [c for c in train_df.columns if train_df[c].dtype != "object" and train_df[c].dtype != "bool"] 
#remove id and is_fraud
numerical_cols.remove("id")
numerical_cols.remove("is_fraud")
numerical_cols

['amount_usd',
 'hours_since_last_txn',
 'txn_count_last_24h',
 'distance_from_home_km',
 'card_age_months',
 'customer_age',
 'account_balance_usd',
 'cvv_retry_count',
 'velocity_score',
 'time_of_day_hour',
 'day_of_week',
 'merchant_risk_score',
 'prior_disputes']

In [36]:
#train_df[numerical_cols].info()

In [44]:
# outlier function
def quantile_outlier(s):
    q1, q3 =  s.quantile([.25, .75])
    iqr=q3-q1
    return (s<q1-1.5*iqr)|(s>q3+1.5*iqr)

def mad_func(s):
    return (s-s.median()).abs().median()

def rz(s):
    mad = mad_func(s)
    return 0.6745*(s-s.median())/mad

quant_outlier = [quantile_outlier(train_df[s]) for s in numerical_cols]
rz_outlier = [rz(train_df[s]) for s in numerical_cols]
outlier = pd.DataFrame({
    "column_name": numerical_cols,
    "quantile_outlier": quant_outlier,
    "rz_outlier": rz_outlier
})

In [47]:
outlier.iloc[0,2]

0          1.768932
1         -0.126241
2         -1.089737
3          0.531567
4         32.686906
            ...    
699955    35.897172
699956     0.262392
699957    -0.224312
699958    -0.681282
699959     0.464795
Name: amount_usd, Length: 699960, dtype: float64